# Methods 4 – Word embeddings

## Objective

The first step is to download the pretrained GloVe model. A word vector represents a word using numbers. In this assignment, I will use pretrained GloVe word vectors to explore how different words are related to each other.

In [1]:

from pathlib import Path
from urllib.request import urlretrieve

url = "https://zenodo.org/records/4925376/files/glove.6B.50d.zip?download=1"
zip_path = Path("glove.6B.50d.zip")

if not zip_path.exists():
    urlretrieve(url, zip_path)

print("Downloaded file:", zip_path.resolve())
print("File size (MB):", round(zip_path.stat().st_size / 1_000_000, 1))

Downloaded file: C:\DEV\Metropolia\neuroverkot\neural_networks\assg 4\glove.6B.50d.zip
File size (MB): 58.9


In [2]:

import zipfile

# Extract the GloVe ZIP file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    print("Files inside ZIP:", zip_ref.namelist())
    zip_ref.extractall()

print("Extraction completed!")


Files inside ZIP: ['glove.6B.50d']
Extraction completed!


## Loading the GloVe embeddings

Here i extracted the Glove embeddings from the zip file. The purpose of this is to access word vectors. 

In [3]:
import numpy as np

word_to_index = {}
vector_dtype = np.dtype("<f4")

with open("glove.6B.50d", "rb") as source:
    word_count, vector_size = map(int, source.readline().split())
    vectors = np.empty((word_count, vector_size), dtype=np.float32)
    vector_bytes = vector_size * vector_dtype.itemsize

    # Each word is followed by a space and a fixed-size binary vector.
    for index in range(word_count):
        word = bytearray()
        while (byte := source.read(1)) != b" ":
            if not byte:
                raise EOFError(f"Missing word at index {index}")
            if byte != b"\n":
                word.extend(byte)

        data = source.read(vector_bytes)
        if len(data) != vector_bytes:
            raise EOFError(f"Incomplete vector at index {index}")
        word_to_index[word.decode("utf-8")] = index
        vectors[index] = np.frombuffer(data, dtype=vector_dtype)

print(f"Loaded {word_count:,} word vectors with {vector_size} dimensions.")

Loaded 400,000 word vectors with 50 dimensions.


As we can see here the model contains 400,000 word vectors, and each vector has 50 dimensions. Next step is to get the vectors for man, woman, and king.

In [5]:
man = vectors[word_to_index["man"]]
woman = vectors[word_to_index["woman"]]
king = vectors[word_to_index["king"]]

print("Man:", man[:5])
print("Woman:", woman[:5])
print("King:", king[:5])

print("Vector length:", len(man))


Man: [-0.094386  0.43007  -0.17224  -0.45529   1.6447  ]
Woman: [-0.18153  0.64827 -0.5821  -0.49451  1.5415 ]
King: [ 0.50451   0.68607  -0.59517  -0.022801  0.60046 ]
Vector length: 50


In [6]:

# Calculate the target vector
target = woman - man + king

# Calculate cosine similarity with all word vectors
target_length = np.linalg.norm(target)
vector_lengths = np.linalg.norm(vectors, axis=1)

similarities = (vectors @ target) / (
    vector_lengths * target_length + 1e-10
)

# Exclude the original words from the results
for word in ["man", "woman", "king"]:
    similarities[word_to_index[word]] = -np.inf

# Find the 10 most similar words
index_to_word = {index: word for word, index in word_to_index.items()}
top_indices = np.argsort(similarities)[-10:][::-1]

for index in top_indices:
    print(index_to_word[index], round(float(similarities[index]), 4))


queen 0.861
daughter 0.7685
prince 0.7641
throne 0.7635
princess 0.7513
elizabeth 0.7506
father 0.7314
kingdom 0.7296
mother 0.728
son 0.728


## Results and explanation

After the lasku wich I did, I can see that the nearest result was queen. kosinisamankaltaisuus was 0,861. The result was logical.

In [7]:

# Function to find words closest to a vector
def find_nearest(target, excluded_words, top_n=5):
    target_norm = np.linalg.norm(target)

    scores = (vectors @ target) / (
        vector_lengths * target_norm + 1e-10
    )

    for word in excluded_words:
        scores[word_to_index[word]] = -np.inf

    top_indices = np.argsort(scores)[-top_n:][::-1]

    return [
        (index_to_word[i], round(float(scores[i]), 4))
        for i in top_indices
    ]


# Test three different word relationships
examples = [
    ("paris", "france", "italy"),
    ("brother", "man", "woman"),
    ("walking", "walk", "swim")
]

for a, b, c in examples:
    target = vectors[word_to_index[a]] - vectors[word_to_index[b]] + vectors[word_to_index[c]]

    results = find_nearest(target, [a, b, c])

    print(f"\n{a} - {b} + {c}")
    for word, similarity in results:
        print(word, similarity)



paris - france + italy
rome 0.8379
milan 0.7715
turin 0.7603
venice 0.7575
madrid 0.7458

brother - man + woman
daughter 0.9374
wife 0.9135
mother 0.8992
father 0.8937
cousin 0.892

walking - walk + swim
swimming 0.8071
swimmers 0.7563
swims 0.748
surfing 0.7419
swam 0.7342


## Other word vector examples

Now in this step, I tried dirrerent exambles such as "paris - france + italy" and others. The result I got was also very logical and intrsting. 
